In [26]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from tqdm import tqdm
tqdm.pandas()
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [27]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')

In [28]:
import wandb

wandb.login(key="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")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [29]:
from typing import Optional
import wandb


class TrainMonitor:

    def __init__(
        self,
        project: str,
        model_name: str,
        version: str = "v1",
        experiment: str = "baseline",
        mode: str = "online",
        config: Optional[dict] = None,
        tags: Optional[list] = None,
        notes: str = "",
        finish_previous: bool = True,
    ):

        self.project = project
        self.model_name = model_name
        self.version = version
        self.experiment = experiment

        self.run = wandb.init(
            project=project,
            name=f"{model_name}_{version}_{experiment}",
            config=config,
            tags=tags,
            notes=notes,
            mode=mode,
            reinit=finish_previous,
        )

    def monitor(self, metrics: dict, step: Optional[int] = None):
        """
        Log any metrics
        """
        wandb.log(metrics, step=step)

    def update_config(self, params: dict):
        wandb.config.update(params, allow_val_change=True)

    def watch(self, model):
        wandb.watch(model)

    def finish(self):
        wandb.finish()

In [42]:
monitor = TrainMonitor(
    project="24f1000781-t22026",
    model_name="fasttext_encoder_MLP",
    version="v3.1",
    experiment="less lr more transformer layers",
    config={
        "lr":3e-4,
        "batch_size":16,
        "epochs":15,
        "embed dim": 256,
        "max length":256
    }
)

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/accuracy,▁▃▅▇▇██████████
train/loss,█▆▄▂▂▂▁▁▁▁▁▁▁▁▁
validation/accuracy,▁▅▇████████████
validation/loss,█▅▄▂▂▁▁▁▁▁▁▁▁▁▁
epoch,14
train/accuracy,0.96222
train/loss,0.09129
validation/accuracy,0.955
validation/loss,0.06173


In [43]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split # ADDED: For validation split
from collections import Counter
import re
import math
import os
import pickle # ADDED: To save the tokenizer state
from gensim.models import FastText # ADDED: Gensim for training custom FastText

# ==========================================
# 1. CUSTOM TOKENIZER & VOCABULARY
# ==========================================
class CustomFastTextTokenizer:
    def __init__(self):
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
        self.vocab_size = 2
        
    def clean_text(self, text):
        text = str(text).lower()
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        return text.split()

    def train_and_build_matrix(self, texts, d_model=128):
        print("Tokenizing corpus for FastText...")
        sentences = [self.clean_text(text) for text in texts]
        
        print(f"Training custom FastText model on {len(sentences)} sequences...")
        # Train FastText. min_count=1 ensures EVERY word gets a vector. 
        # FastText's n-gram feature will handle any weird variations.
        ft_model = FastText(sentences=sentences, vector_size=d_model, window=5, min_count=1, workers=4, epochs=15)
        
        # Build vocabulary from the trained model
        words = list(ft_model.wv.index_to_key)
        
        # Initialize embedding matrix (vocab_size + 2 for PAD and UNK)
        self.vocab_size = len(words) + 2
        embedding_matrix = np.zeros((self.vocab_size, d_model))
        
        # UNK token gets random initialization, PAD stays 0
        embedding_matrix[1] = np.random.normal(scale=0.1, size=(d_model,))
        
        print("Transferring weights to PyTorch embedding matrix...")
        for i, word in enumerate(words):
            idx = i + 2 # Shift by 2 because 0=PAD, 1=UNK
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            embedding_matrix[idx] = ft_model.wv[word]
            
        print(f"Custom FastText Vocabulary built with {self.vocab_size} tokens.")
        return torch.tensor(embedding_matrix, dtype=torch.float32)

    def encode(self, text, max_len):
        words = self.clean_text(text)
        tokens = [self.word2idx.get(w, 1) for w in words] # 1 is <UNK>
        
        # Truncate or Pad
        if len(tokens) > max_len:
            tokens = tokens[:max_len]
        else:
            tokens = tokens + [0] * (max_len - len(tokens)) # 0 is <PAD>
        return tokens

# ==========================================
# 2. DATASET BUILDER
# ==========================================
class ScratchMCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        # We encode [Prompt + Option] for all 5 choices
        input_ids = []
        for opt in self.options:
            combined_text = prompt + " " + str(row[opt])
            encoded = self.tokenizer.encode(combined_text, self.max_len)
            input_ids.append(encoded)
            
        item = {
            'input_ids': torch.tensor(input_ids, dtype=torch.long) # Shape: (5, max_len)
        }
        
        if not self.is_test:
            ans_idx = self.options.index(row['answer'])
            item['label'] = torch.tensor(ans_idx, dtype=torch.long)
            
        return item

# ==========================================
# 3. MICRO-TRANSFORMER ARCHITECTURE
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (seq_len, batch_size, d_model)
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class TinyMCQModel(nn.Module):
    def __init__(self, vocab_size, d_model=300, nhead=6, num_layers=1, dropout=0.3, pretrained_embeddings=None):
        super().__init__()
        # Load Pre-trained FastText weights if provided
        if pretrained_embeddings is not None:
            # freeze=False allows the model to fine-tune the FastText embeddings specifically for this task
            self.embedding = nn.Embedding.from_pretrained(pretrained_embeddings, freeze=False, padding_idx=0)
        else:
            self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
            
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        encoder_layers = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=nhead, 
            dim_feedforward=512, 
            dropout=dropout,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        
        # Projects the encoded sequence to a single score
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, input_ids):
        # input_ids: (batch_size, 5, seq_len)
        batch_size, num_opts, seq_len = input_ids.shape
        
        # Flatten options into batch dimension: (batch_size * 5, seq_len)
        x = input_ids.view(batch_size * num_opts, seq_len)
        
        # Embed and encode
        x = self.embedding(x) # (batch_size * 5, seq_len, d_model)
        x = self.transformer_encoder(x) # (batch_size * 5, seq_len, d_model)
        
        # Global Average Pooling (ignore padding in a real scenario, simplified here)
        x = x.mean(dim=1) # (batch_size * 5, d_model)
        
        # Score each option
        logits = self.classifier(x) # (batch_size * 5, 1)
        
        # Reshape back to (batch_size, 5)
        logits = logits.view(batch_size, num_opts)
        return logits

# ==========================================
# 4. TRAINING & INFERENCE PIPELINE
# ==========================================
def run_scratch_pipeline(train_df: pd.DataFrame, test_df: pd.DataFrame, save_dir: str = None):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # ADDED: 10% Validation Split
    train_df, val_df = train_test_split(train_df, test_size=0.1, random_state=42)
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    # 1. Build Tokenizer and Train FastText from ALL available text (Train + Val + Test)
    tokenizer = CustomFastTextTokenizer()
    
    # Gathering corpus for unsupervised learning
    all_text = train_df['prompt'].tolist() + val_df['prompt'].tolist() + test_df['prompt'].tolist()
    for opt in ['A', 'B', 'C', 'D', 'E']:
        all_text.extend(train_df[opt].tolist())
        all_text.extend(val_df[opt].tolist())
        all_text.extend(test_df[opt].tolist())
        
    # We use d_model=128 because the dataset is small. 300 dimensions might lead to overfitting 
    # unless you have hundreds of thousands of rows.
    D_MODEL = 256 
    pretrained_embeddings = tokenizer.train_and_build_matrix(all_text, d_model=D_MODEL)

    # 2. Datasets & DataLoaders
    max_len = 256
    train_ds = ScratchMCQDataset(train_df, tokenizer, max_len=max_len)
    val_ds = ScratchMCQDataset(val_df, tokenizer, max_len=max_len) # ADDED: Validation Dataset
    test_ds = ScratchMCQDataset(test_df, tokenizer, max_len=max_len, is_test=True)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False) # ADDED: Validation Loader
    test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)

    # 3. Initialize Model, Loss, Optimizer
    model = TinyMCQModel(
        vocab_size=tokenizer.vocab_size, 
        d_model=D_MODEL, # Must match FastText dimension
        nhead=4,         # 128 is divisible by 4
        num_layers=2, 
        dropout=0.3,
        pretrained_embeddings=pretrained_embeddings
    ).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)

    # 4. Training Loop
    epochs = 15 # Increased to 25 based on established performance peak
    print(f"\nStarting Training for {epochs} Epochs...")
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        total_train_loss = 0
        train_correct = 0
        train_total = 0
        
        for batch in train_loader:
            inputs = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            optimizer.zero_grad()
            logits = model(inputs) # (batch_size, 5)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
        train_loss = total_train_loss / len(train_loader)
        train_acc = train_correct / train_total

        # --- VALIDATION PHASE ---
        model.eval()
        total_val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['input_ids'].to(device)
                labels = batch['label'].to(device)
                
                logits = model(inputs)
                loss = criterion(logits, labels)
                
                total_val_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = total_val_loss / len(val_loader) if len(val_loader) > 0 else 0.0
        val_acc = val_correct / val_total if val_total > 0 else 0.0

        try:
            monitor.monitor({
                "epoch": epoch,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "validation/loss": val_loss,
                "validation/accuracy": val_acc,
            })
        except NameError:
            pass 

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # ADDED: Save the model and tokenizer if a directory is provided
    if save_dir:
        print(f"\nSaving model and tokenizer to {save_dir}...")
        os.makedirs(save_dir, exist_ok=True)
        # Save PyTorch weights
        torch.save(model.state_dict(), os.path.join(save_dir, "tiny_mcq_model.pth"))
        # Save Tokenizer (Vocabulary)
        with open(os.path.join(save_dir, "tokenizer.pkl"), "wb") as f:
            pickle.dump(tokenizer, f)
        print("Save complete!")

    # 5. Inference
    print("\nGenerating Test Predictions...")
    model.eval()
    all_preds = []
    options = ['A', 'B', 'C', 'D', 'E']
    
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['input_ids'].to(device)
            logits = model(inputs)
            
            # Sort to get top 3 indices
            scores = logits.cpu().numpy()
            top_3_idx = np.argsort(scores, axis=1)[:, ::-1][:, :3]
            
            for row in top_3_idx:
                pred_str = " ".join([options[i] for i in row])
                all_preds.append(pred_str)

    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': all_preds
    })
    
    return submission_df

In [44]:
sub_df = run_scratch_pipeline(train_df,test_df,'/kaggle/working/model')

Using device: cuda
Tokenizing corpus for FastText...
Training custom FastText model on 15000 sequences...
Transferring weights to PyTorch embedding matrix...
Custom FastText Vocabulary built with 2975 tokens.

Starting Training for 15 Epochs...
Epoch 1/15 | Train Loss: 1.4397 | Train Acc: 0.3539 | Val Loss: 1.1204 | Val Acc: 0.5800
Epoch 2/15 | Train Loss: 0.9425 | Train Acc: 0.6061 | Val Loss: 0.5487 | Val Acc: 0.8300
Epoch 3/15 | Train Loss: 0.5105 | Train Acc: 0.7961 | Val Loss: 0.2411 | Val Acc: 0.9400
Epoch 4/15 | Train Loss: 0.2978 | Train Acc: 0.8867 | Val Loss: 0.1599 | Val Acc: 0.9500
Epoch 5/15 | Train Loss: 0.1919 | Train Acc: 0.9228 | Val Loss: 0.1354 | Val Acc: 0.9400
Epoch 6/15 | Train Loss: 0.1494 | Train Acc: 0.9461 | Val Loss: 0.1256 | Val Acc: 0.9450
Epoch 7/15 | Train Loss: 0.1236 | Train Acc: 0.9506 | Val Loss: 0.0920 | Val Acc: 0.9550
Epoch 8/15 | Train Loss: 0.1218 | Train Acc: 0.9511 | Val Loss: 0.0864 | Val Acc: 0.9850
Epoch 9/15 | Train Loss: 0.0950 | Train Acc

In [33]:
sub_df.to_csv('submission.csv',index=False)

In [41]:
sub_df

,id,Prediction
0,1,A E B
1,2,B C A
2,3,B E D
3,4,E A D
4,5,C D B
...,...,...
495,496,A B E
496,497,C B E
497,498,B C E
498,499,E B D


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.
